# Stage 3: Model Training & Evaluation
## Multinomial Naive Bayes for Support Ticket Classification

**Project:** naive-bayes-tickets  
**Team Member:** Person 3 (Model Training & Evaluation)  
**Date:** 2026  

### Overview
This notebook trains and evaluates a Multinomial Naive Bayes classifier on processed support ticket features. 
Key tasks:
- Load preprocessed features (X_train, X_test, y_train, y_test)
- Hyperparameter tuning with cross-validation
- Model evaluation and performance analysis
- Confusion matrix visualization
- Save trained model and create inference pipeline

In [ ]:
# Import Required Libraries
import os
import sys
import warnings
import pickle
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import load_npz
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Section 1: Load Processed Features and Artifacts

Load the .npz and .npy files from `data/features/` containing X_train, X_test, y_train, y_test. 
Also load the label encoder, TF-IDF vectorizer, and MinMaxScaler from `src/` for post-training inference.

In [ ]:
# Define paths
PROJ_ROOT  = Path('../').resolve()
DATA_FEAT_DIR = PROJ_ROOT / 'data' / 'features'
SRC_DIR    = PROJ_ROOT / 'src'
FIGURES_DIR = PROJ_ROOT / 'figures'

# Ensure directories exist
FIGURES_DIR.mkdir(exist_ok=True)

logger.info(f"Project root: {PROJ_ROOT}")
logger.info(f"Features directory: {DATA_FEAT_DIR}")
logger.info(f"Source directory: {SRC_DIR}")

# Load feature matrices
logger.info("Loading feature matrices...")
X_train = load_npz(DATA_FEAT_DIR / 'X_train.npz')
X_test  = load_npz(DATA_FEAT_DIR / 'X_test.npz')
y_train = np.load(DATA_FEAT_DIR / 'y_train.npy')
y_test  = np.load(DATA_FEAT_DIR / 'y_test.npy')

logger.info(f"X_train shape: {X_train.shape}")
logger.info(f"X_test shape:  {X_test.shape}")
logger.info(f"y_train shape: {y_train.shape}")
logger.info(f"y_test shape:  {y_test.shape}")

# Load preprocessing artifacts
logger.info("Loading preprocessing artifacts...")
with open(SRC_DIR / 'label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

with open(SRC_DIR / 'tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)

with open(SRC_DIR / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

logger.info("All artifacts loaded successfully!")

# Display class mapping
class_names = label_encoder.classes_
logger.info(f"Class mapping: {dict(enumerate(class_names))}")

## Section 2: Train Multinomial Naive Bayes with Hyperparameter Tuning

Use GridSearchCV with cross-validation to tune the alpha hyperparameter (Laplace smoothing) 
over values [0.01, 0.1, 0.5, 1.0, 2.0]. The best alpha will be selected based on cross-validation performance.

In [ ]:
# Define hyperparameter grid
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
}

logger.info("Starting hyperparameter tuning with GridSearchCV...")
logger.info(f"Parameters to search: {param_grid}")

# Initialize base model
mnb = MultinomialNB()

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    estimator=mnb,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
logger.info("Fitting model with cross-validation...")
grid_search.fit(X_train, y_train)

# Extract best parameters
best_alpha = grid_search.best_params_['alpha']
best_score = grid_search.best_score_

logger.info(f"\n{'='*60}")
logger.info(f"Best alpha: {best_alpha}")
logger.info(f"Best cross-validation F1-score (weighted): {best_score:.4f}")
logger.info(f"{'='*60}\n")

# Display all cross-validation results
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results_display = cv_results[['param_alpha', 'mean_test_score', 'std_test_score']].copy()
cv_results_display.columns = ['Alpha', 'Mean CV F1-Score', 'Std CV F1-Score']
print("\nCross-Validation Results:")
print(cv_results_display.to_string(index=False))

# Get best model
best_model = grid_search.best_estimator_
logger.info(f"Best model: {best_model}")

## Section 3: Evaluate Model Performance

Generate detailed evaluation metrics on the test set including:
- Overall accuracy
- Classification report (precision, recall, F1-score per class)
- Per-class performance analysis

In [ ]:
# Make predictions
logger.info("Making predictions on test set...")
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Calculate accuracy scores
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

logger.info(f"\nAccuracy Scores:")
logger.info(f"  Train Accuracy: {train_accuracy:.4f}")
logger.info(f"  Test Accuracy:  {test_accuracy:.4f}")

# Classification Report
logger.info(f"\n{'='*60}")
logger.info("CLASSIFICATION REPORT (Test Set)")
logger.info(f"{'='*60}")
class_report = classification_report(
    y_test, 
    y_pred_test, 
    target_names=class_names,
    digits=4
)
print(class_report)

# Detailed per-class metrics
logger.info(f"\n{'='*60}")
logger.info("PER-CLASS PERFORMANCE SUMMARY")
logger.info(f"{'='*60}")

per_class_metrics = []
for i, class_name in enumerate(class_names):
    precision = precision_score(y_test, y_pred_test, labels=[i], average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred_test, labels=[i], average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred_test, labels=[i], average='weighted', zero_division=0)
    
    per_class_metrics.append({
        'Class': class_name,
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-Score': f'{f1:.4f}'
    })

metrics_df = pd.DataFrame(per_class_metrics)
print("\n" + metrics_df.to_string(index=False))

## Section 4: Generate Confusion Matrix Visualization

Visualize the confusion matrix to identify which classes are most frequently misclassified
and where the model struggles most.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred_test)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'},
    ax=ax
)

ax.set_title('Confusion Matrix - Multinomial Naive Bayes\n(Test Set)', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)

# Rotate labels for better readability
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
confusion_matrix_path = FIGURES_DIR / 'confusion_matrix.png'
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches='tight')
logger.info(f"Confusion matrix saved to: {confusion_matrix_path}")
plt.show()

# Print confusion matrix analysis
logger.info(f"\n{'='*60}")
logger.info("CONFUSION MATRIX ANALYSIS")
logger.info(f"{'='*60}")
for i, class_name in enumerate(class_names):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy_per_class = correct / total * 100 if total > 0 else 0
    logger.info(f"{class_name}: {correct}/{total} correct ({accuracy_per_class:.2f}%)")

## Section 5: Save Trained Model and Inference Artifacts

Save the trained MultinomialNB model and verify all preprocessing artifacts 
are ready for the inference pipeline.

In [ ]:
# Save the trained model
logger.info("Saving trained model...")
model_path = SRC_DIR / 'naive_bayes_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

logger.info(f"Model saved to: {model_path}")

# Verify all artifacts exist
logger.info(f"\n{'='*60}")
logger.info("VERIFYING ALL ARTIFACTS")
logger.info(f"{'='*60}")

artifacts = {
    'Model': model_path,
    'Label Encoder': SRC_DIR / 'label_encoder.pkl',
    'TF-IDF Vectorizer': SRC_DIR / 'tfidf_vectorizer.pkl',
    'MinMaxScaler': SRC_DIR / 'scaler.pkl'
}

all_artifacts_exist = True
for artifact_name, artifact_path in artifacts.items():
    exists = artifact_path.exists()
    status = '✓' if exists else '✗'
    logger.info(f"{status} {artifact_name}: {artifact_path}")
    if not exists:
        all_artifacts_exist = False

if all_artifacts_exist:
    logger.info("\n✓ All artifacts are ready for inference!")
else:
    logger.warning("\n✗ Some artifacts are missing!")

## Section 6: Create Prediction Pipeline for New Tickets

Build a reusable function that takes a raw ticket description and returns the predicted class
with confidence scores. This function can be used for inference in production.

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK data (run once)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Initialize preprocessing components
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_ticket(text: str) -> str:
    """
    Preprocess a ticket description following the same pipeline as training data.
    
    Steps:
    1. Convert to lowercase
    2. Remove URLs and emails
    3. Remove alphanumeric IDs
    4. Remove punctuation
    5. Tokenize
    6. Remove stopwords (but keep negations)
    7. Lemmatize
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove alphanumeric IDs (sequences of letters/numbers)
    text = re.sub(r'[a-z0-9]*[a-z][a-z0-9]*\d+|[a-z0-9]*\d+[a-z0-9]*[a-z]', '', text)
    
    # Remove punctuation but keep spaces
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords (but keep negations)
    negation_words = {'not', 'no', 'nor', 'neither', 'never', 'cannot', 'can\'t', 'won\'t'}
    tokens = [token for token in tokens if token not in stop_words or token in negation_words]
    
    # Lemmatize
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Rejoin tokens
    cleaned_text = ' '.join(tokens)
    
    return cleaned_text

def predict_ticket_class(raw_description: str) -> dict:
    """
    Predict the class of a support ticket.
    
    Args:
        raw_description: Raw ticket description text
        
    Returns:
        Dictionary with:
        - 'predicted_class': Predicted ticket class
        - 'class_id': Numeric class ID
        - 'confidence': Probability of predicted class
        - 'probabilities': Dict of all class probabilities
    """
    # Preprocess
    cleaned_description = preprocess_ticket(raw_description)
    
    # Vectorize with TF-IDF
    X_tfidf = tfidf_vectorizer.transform([cleaned_description])
    
    # Get predictions and probabilities
    class_id = best_model.predict(X_tfidf)[0]
    class_proba = best_model.predict_proba(X_tfidf)[0]
    
    # Get predicted class name
    predicted_class = label_encoder.inverse_transform([class_id])[0]
    confidence = class_proba[class_id]
    
    # Create probability dictionary
    probabilities = {
        class_names[i]: float(class_proba[i])
        for i in range(len(class_names))
    }
    
    return {
        'predicted_class': predicted_class,
        'class_id': int(class_id),
        'confidence': float(confidence),
        'probabilities': probabilities,
        'cleaned_description': cleaned_description
    }

logger.info("Prediction pipeline created successfully!")

## Test: Prediction Pipeline with Sample Tickets

Test the prediction pipeline with example tickets to verify it works correctly.

In [ ]:
# Test with sample tickets
sample_tickets = [
    "I need a refund for my recent purchase. The product did not meet my expectations.",
    "My software keeps crashing when I try to open large files. Can you help troubleshoot?",
    "I want to cancel my subscription. I'm no longer interested in the service.",
    "Do you have this product in a different color? I'm interested in the blue variant.",
    "I was overcharged on my billing invoice. Please review my account charges."
]

logger.info(f"\n{'='*80}")
logger.info("PREDICTION PIPELINE TEST - Sample Tickets")
logger.info(f"{'='*80}\n")

for i, ticket in enumerate(sample_tickets, 1):
    result = predict_ticket_class(ticket)
    
    print(f"Ticket {i}:")
    print(f"  Description: {ticket[:60]}...")
    print(f"  Predicted Class: {result['predicted_class']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print(f"  Probabilities:")
    for class_name, prob in sorted(result['probabilities'].items(), key=lambda x: x[1], reverse=True):
        print(f"    - {class_name}: {prob:.2%}")
    print()